# Calories Burnt Prediction AI - Model Training Notebook
This notebook walks through the same pipeline as `src/train_ml.py` and `src/train_dl.py`, step by step, so each stage can be explained in a viva.

Pipeline: **Load -> Clean -> EDA -> Split -> Scale -> Train -> Evaluate -> Compare**

In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


## 1. Load and clean the data

In [7]:
df = pd.read_csv(r"C:\Users\mrudu\OneDrive\Desktop\calories-burnt-ai\data\activity_catalog.csv")
df = pd.read_csv(r"C:\Users\mrudu\OneDrive\Desktop\calories-burnt-ai\data\demo_workouts.csv")
df = pd.read_csv(r"C:\Users\mrudu\OneDrive\Desktop\calories-burnt-ai\data\raw_calories.csv")

In [8]:
df.shape

(15000, 2)

In [9]:
df.columns

Index(['User_ID', 'Calories'], dtype='str')

In [10]:
df.describe()

,User_ID,Calories
count,1.500000e+04,15000.000000
mean,1.497736e+07,89.539533
std,2.872851e+06,62.456978
min,1.000116e+07,1.000000
25%,1.247419e+07,35.000000
50%,1.499728e+07,79.000000
75%,1.744928e+07,138.000000
max,1.999965e+07,314.000000


In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   User_ID   15000 non-null  int64  
 1   Calories  15000 non-null  float64
dtypes: float64(1), int64(1)
memory usage: 234.5 KB


In [12]:
df.isnull().sum()

User_ID     0
Calories    0
dtype: int64

In [13]:
df.duplicated()

0        False
1        False
2        False
3        False
4        False
         ...  
14995    False
14996    False
14997    False
14998    False
14999    False
Length: 15000, dtype: bool

In [14]:
def clean_column_names(df):
    """
    Standardize column names.
    """

    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("-", "_")
    )

    return df

In [15]:
def remove_duplicates(df):
    """
    Remove duplicate records.
    """

    before = len(df)

    df = df.drop_duplicates()

    after = len(df)

    print(f"\nDuplicates removed: {before - after}")

    return df

In [16]:
def handle_missing_values(df):
    """
    Handle missing values.

    Numerical columns -> median
    Categorical columns -> mode
    """

    numerical_columns = df.select_dtypes(
        include=[np.number]
    ).columns

    categorical_columns = df.select_dtypes(
        exclude=[np.number]
    ).columns

    # Numerical missing values
    for column in numerical_columns:
        if df[column].isnull().sum() > 0:
            df[column] = df[column].fillna(
                df[column].median()
            )

    # Categorical missing values
    for column in categorical_columns:
        if df[column].isnull().sum() > 0:
            df[column] = df[column].fillna(
                df[column].mode()[0]
            )

    return df

In [17]:
def remove_invalid_values(df):
    """
    Remove invalid numerical values.
    """

    numerical_columns = df.select_dtypes(
        include=[np.number]
    ).columns

    for column in numerical_columns:
        df = df[df[column] >= 0]

    return df

In [18]:
def clean_data(df):

    print("\n========== CLEANING DATA ==========")

    # Clean column names
    df = clean_column_names(df)

    # Remove duplicates
    df = remove_duplicates(df)

    # Handle missing values
    df = handle_missing_values(df)

    # Remove invalid values
    df = remove_invalid_values(df)

    # Reset index
    df = df.reset_index(drop=True)

    print("\nCleaning completed!")
    print(f"Final dataset shape: {df.shape}")

    print("\nRemaining missing values:")
    print(df.isnull().sum())

    return df

## 3. Train / test split and feature scaling
The scaler is fitted on the training set only to avoid data leakage.  `save_artifacts=False` keeps this notebook from overwriting the app's saved files.

In [23]:
df = pd.read_csv(r"C:\Users\mrudu\OneDrive\Desktop\calories-burnt-ai\data\activity_catalog.csv") 
df = pd.read_csv(r"C:\Users\mrudu\OneDrive\Desktop\calories-burnt-ai\data\demo_workouts.csv")
df = pd.read_csv(r"C:\Users\mrudu\OneDrive\Desktop\calories-burnt-ai\data\raw_calories.csv")
print("Dataset loaded successfully!") 
print("Dataset shape:", df.shape)

Dataset loaded successfully!
Dataset shape: (15000, 2)


In [24]:
if "gender" in df.columns: 
    df["gender"] = df["gender"].map({
         "Male": 1,
         "Female": 0
   })

In [25]:
columns_to_remove = [] 
if "id" in df.columns: 
    columns_to_remove.append("id") 
if "user_id" in df.columns: 
    columns_to_remove.append("user_id") 
df = df.drop( 
    columns=columns_to_remove, 
    errors="ignore" 
)

In [27]:
# Normalize column names to avoid case/spacing issues like "Calories" vs "calories"
df.columns = df.columns.str.strip().str.lower()

# Drop target column safely
X = df.drop(columns=["calories"], errors="ignore")

y = df["calories"]
print("\nFeatures:")
print(X.columns.tolist())
print("\nTarget:")
print("calories")
y = df["calories"] 
print("\nFeatures:") 
print(X.columns.tolist()) 
print("\nTarget:") 
print("calories")


Features:
['user_id']

Target:
calories

Features:
['user_id']

Target:
calories


In [28]:
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.20, random_state=42 )

In [29]:
print("\n========== TRAIN-TEST SPLIT ==========") 
print("Training samples:", X_train.shape[0]) 
print("Testing samples:", X_test.shape[0]) 
print("Training features:", X_train.shape[1]) 
print("Testing features:", X_test.shape[1])


========== TRAIN-TEST SPLIT ==========
Training samples: 12000
Testing samples: 3000
Training features: 1
Testing features: 1


In [30]:
scaler = StandardScaler()  
# Fit only on training data 
# Then transform both training and testing data 
X_train_scaled = scaler.fit_transform(X_train) 
X_test_scaled = scaler.transform(X_test)

In [31]:
X_train_scaled = pd.DataFrame(
     X_train_scaled, 
     columns=X_train.columns, 
     index=X_train.index 
     )
X_test_scaled = pd.DataFrame( 
    X_test_scaled, 
    columns=X_test.columns, 
    index=X_test.index 
    )

In [32]:
print("\n========== BEFORE SCALING ==========") 
print(X_train.head()) 
print("\n========== AFTER SCALING ==========")
print(X_train_scaled.head())


========== BEFORE SCALING ==========
        user_id
9839   16554569
9680   18903739
7093   11938260
11293  14116395
820    13815395

========== AFTER SCALING ==========
        user_id
9839   0.550991
9680   1.368600
7093  -1.055676
11293 -0.297594
820   -0.402355


In [33]:
print("\n========== SCALED DATA STATISTICS ==========") 
print( X_train_scaled.describe() )


========== SCALED DATA STATISTICS ==========
            user_id
count  1.200000e+04
mean  -2.960595e-16
std    1.000042e+00
min   -1.729867e+00
25%   -8.777652e-01
50%    5.963429e-03
75%    8.623872e-01
max    1.749933e+00


In [35]:
from pathlib import Path

Path("data").mkdir(parents=True, exist_ok=True)

X_train_scaled.to_csv( 
    "data/X_train_scaled.csv", 
    index=False 
    )
X_test_scaled.to_csv( 
    "data/X_test_scaled.csv", 
    index=False 
    )
y_train.to_csv( 
    "data/y_train.csv", 
    index=False 
    )
y_test.to_csv( 
    "data/y_test.csv", 
    index=False 
)
print("\nProcessed datasets saved successfully!") 
print("\n========== PREPROCESSING COMPLETED ==========")


Processed datasets saved successfully!

========== PREPROCESSING COMPLETED ==========


## 4. Machine-learning models

In [36]:
from sklearn.linear_model import LinearRegression 
from sklearn.ensemble import RandomForestRegressor 
from sklearn.ensemble import GradientBoostingRegressor 
from sklearn.metrics import (
     mean_absolute_error,
     mean_squared_error, 
     r2_score 
     )

In [37]:
models = { 
    "Linear Regression": LinearRegression(), 
    "Random Forest": RandomForestRegressor( 
        n_estimators=200, 
        random_state=42, 
        n_jobs=-1 
        ), 
        "Gradient Boosting": GradientBoostingRegressor( 
            n_estimators=200, 
            learning_rate=0.05, 
            max_depth=3, 
            random_state=42 
        ) 
}

In [38]:
results = [] 
trained_models = {} 
for name, model in models.items(): 
    print("\n" + "=" * 50) 
    print(f"Training: {name}") 
    print("=" * 50) 
    # Train 
    model.fit(X_train, y_train) 
    # Prediction
    y_pred = model.predict(X_test) 
    # # Metrics
    mae = mean_absolute_error(
         y_test, 
         y_pred 
         ) 
    mse = mean_squared_error(
         y_test, 
         y_pred 
         ) 
    rmse = np.sqrt(mse) 
    r2 = r2_score(
         y_test, 
         y_pred 
         ) 
    # Store results 
    results.append({
         "Model": name, 
         "MAE": mae, 
         "MSE": mse, 
         "RMSE": rmse, 
         "R2 Score": r2 
         }) 
    trained_models[name] = model 
    print(f"MAE : {mae:.4f}") 
    print(f"MSE : {mse:.4f}") 
    print(f"RMSE : {rmse:.4f}") 
    print(f"R2 Score : {r2:.4f}")
     


Training: Linear Regression
MAE : 53.9336
MSE : 4041.5723
RMSE : 63.5734
R2 Score : -0.0014

Training: Random Forest
MAE : 62.8597
MSE : 6021.9275
RMSE : 77.6011
R2 Score : -0.4921

Training: Gradient Boosting
MAE : 53.9859
MSE : 4052.8173
RMSE : 63.6617
R2 Score : -0.0042


In [39]:
results_df = pd.DataFrame(results) 
results_df = results_df.sort_values( by="R2 Score", ascending=False ) 
print("\n\n========== MODEL COMPARISON ==========") 
print( results_df.to_string(index=False) )



========== MODEL COMPARISON ==========
            Model       MAE         MSE      RMSE  R2 Score
Linear Regression 53.933648 4041.572316 63.573362 -0.001432
Gradient Boosting 53.985894 4052.817310 63.661741 -0.004219
    Random Forest 62.859667 6021.927506 77.601079 -0.492130


In [40]:
best_model_name = results_df.iloc[0]["Model"] 
best_model = trained_models[ best_model_name ] 
print("\n========================================") 
print("BEST MODEL") 
print("========================================") 
print("Model:", best_model_name) 
print( "R2 Score:", results_df.iloc[0]["R2 Score"] ) 
print( "RMSE:", results_df.iloc[0]["RMSE"] ) 
print( "MAE:", results_df.iloc[0]["MAE"] )


BEST MODEL
Model: Linear Regression
R2 Score: -0.0014323927992658803
RMSE: 63.57336168541667
MAE: 53.93364822763257


In [42]:
import joblib

In [44]:
from pathlib import Path

Path("models").mkdir(parents=True, exist_ok=True)
joblib.dump(best_model, "models/best_calories_model.pkl")
print("\nBest model saved as: models/best_calories_model.pkl")
print( "\nBest model saved as:" " models/best_calories_model.pkl" )


Best model saved as: models/best_calories_model.pkl

Best model saved as: models/best_calories_model.pkl


In [45]:
results_df.to_csv( 
    "models/model_comparison.csv", index=False ) 
print( "Model comparison saved as:" 
                                                                      " models/model_comparison.csv" )

Model comparison saved as: models/model_comparison.csv


In [46]:
sample_predictions = best_model.predict( 
    X_test.head(10) ) 
print("\n========== SAMPLE PREDICTIONS ==========") 
for actual, predicted in zip( y_test.head(10), 
sample_predictions ): 
    print( f"Actual: {actual:.2f} | " 
                                                                                                                                                                                          f"Predicted: {predicted:.2f}" )


========== SAMPLE PREDICTIONS ==========
Actual: 173.00 | Predicted: 88.88
Actual: 189.00 | Predicted: 89.01
Actual: 53.00 | Predicted: 89.30
Actual: 161.00 | Predicted: 89.38
Actual: 226.00 | Predicted: 89.35
Actual: 179.00 | Predicted: 89.22
Actual: 98.00 | Predicted: 89.47
Actual: 44.00 | Predicted: 89.38
Actual: 79.00 | Predicted: 88.97
Actual: 59.00 | Predicted: 89.37


## 5. Deep-learning model (Dense neural network)

In [47]:
import pandas as pd 
import numpy as np 
import joblib 
import os 
import tensorflow as tf 
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Dense, Dropout 
from tensorflow.keras.callbacks import EarlyStopping

In [48]:
from sklearn.metrics import ( 
    mean_absolute_error,
      mean_squared_error, 
      r2_score 
      )

In [49]:
X_train = pd.read_csv( "data/X_train_scaled.csv" ) 
X_test = pd.read_csv( "data/X_test_scaled.csv" ) 
y_train = pd.read_csv( "data/y_train.csv" ).squeeze() 
y_test = pd.read_csv( "data/y_test.csv" ).squeeze() 
print("Data loaded successfully!") 
print("X_train shape:", X_train.shape) 
print("X_test shape :", X_test.shape)

Data loaded successfully!
X_train shape: (12000, 1)
X_test shape : (3000, 1)


In [50]:
model = Sequential([ 
    # Input layer + first hidden layer 
    Dense( 128, activation="relu", input_shape=(X_train.shape[1],)
           ), 
           # Dropout helps reduce overfitting 
           Dropout(0.20), 
           # Second hidden layer 
           Dense( 64, activation="relu" ), Dropout(0.20), 
           # Third hidden layer 
           Dense( 32, activation="relu" ), 
           # Output layer # One neuron because calories is a continuous value 
           Dense( 1, activation="linear" ) ])

c:\Users\mrudu\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:102: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [51]:
print("\n========== MODEL ARCHITECTURE ==========")
model.summary()


========== MODEL ARCHITECTURE ==========


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,625 (41.50 KB)

 Trainable params: 10,625 (41.50 KB)

 Non-trainable params: 0 (0.00 B)

In [52]:
model.compile( 
    optimizer="adam", 
    loss="mse", 
    metrics=["mae"] )

In [53]:
early_stopping = EarlyStopping( 
    monitor="val_loss", 
    patience=15, 
    restore_best_weights=True 
    )

In [55]:
print("\n========== TRAINING MODEL ==========") 
history = model.fit( 
    X_train, y_train,
      validation_split=0.20, 
      epochs=20, 
      batch_size=32, 
      callbacks=[early_stopping], 
      verbose=1 )


========== TRAINING MODEL ==========
Epoch 1/20
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 3914.6445 - mae: 52.8591 - val_loss: 3976.8359 - val_mae: 53.9269
Epoch 2/20
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 3933.3049 - mae: 52.9508 - val_loss: 3975.5183 - val_mae: 53.8625
Epoch 3/20
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 3924.3508 - mae: 52.9427 - val_loss: 3974.8804 - val_mae: 53.8290
Epoch 4/20
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 3930.8892 - mae: 52.9887 - val_loss: 3974.1272 - val_mae: 53.8219
Epoch 5/20
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 3916.1021 - mae: 52.9749 - val_loss: 3975.8674 - val_mae: 53.8916
Epoch 6/20
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 3929.5400 - mae: 52.9966 - val_loss: 3974.9812 - val_mae: 53.8769
Epoch 7/20
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 3918.0916 - mae: 52.9271 - val_loss: 3976.5107 - val_mae: 53.9598
Epoch 8/20
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 3927.8479 - mae: 52.

In [57]:
print("\n========== MODEL EVALUATION ==========") 
test_loss, test_mae = model.evaluate(
    X_test,
    y_test,
    verbose=0
)
print( f"Test Loss (MSE): {test_loss:.4f}" ) 
print( f"Test MAE: {test_mae:.4f}" )


========== MODEL EVALUATION ==========
Test Loss (MSE): 4043.4067
Test MAE: 54.0719


In [58]:
y_pred = model.predict( X_test, verbose=0 ).flatten()

In [59]:
mae = mean_absolute_error( y_test, y_pred ) 
mse = mean_squared_error( y_test, y_pred ) 
rmse = np.sqrt(mse) 
r2 = r2_score( y_test, y_pred ) 
print("\n========== DEEP LEARNING RESULTS ==========") 
print(f"MAE : {mae:.4f}") 
print(f"MSE : {mse:.4f}") 
print(f"RMSE : {rmse:.4f}") 
print(f"R2 Score : {r2:.4f}")


========== DEEP LEARNING RESULTS ==========
MAE : 54.0719
MSE : 4043.4067
RMSE : 63.5878
R2 Score : -0.0019


In [60]:
os.makedirs( "models", exist_ok=True )

In [61]:
model.save( "models/calories_dense_model.keras" ) 
print( "\nDeep Learning model saved successfully!" ) 
print( "Location: models/calories_dense_model.keras" )


Deep Learning model saved successfully!
Location: models/calories_dense_model.keras


In [62]:
history_df = pd.DataFrame( history.history ) 
history_df.to_csv( "models/training_history.csv", index=False ) 
print( "Training history saved successfully!" )

Training history saved successfully!


In [64]:
print("\n========== SAMPLE PREDICTIONS ==========") 
for actual, predicted in zip( 
    y_test.iloc[:10], 
    y_pred[:10] 
): 
    print( 
        f"Actual: {actual:.2f} | " 
        f"Predicted: {predicted:.2f}" 
    ) 
print("\n========== TRAINING COMPLETED ==========")


========== SAMPLE PREDICTIONS ==========
Actual: 173.00 | Predicted: 89.13
Actual: 189.00 | Predicted: 89.50
Actual: 53.00 | Predicted: 90.25
Actual: 161.00 | Predicted: 90.88
Actual: 226.00 | Predicted: 90.46
Actual: 179.00 | Predicted: 90.09
Actual: 98.00 | Predicted: 92.50
Actual: 44.00 | Predicted: 90.92
Actual: 79.00 | Predicted: 89.40
Actual: 59.00 | Predicted: 90.75

========== TRAINING COMPLETED ==========


## 6. Best model
The model with the lowest RMSE on the test set is used by the app when *Best model (auto)* is selected.

In [65]:
import pandas as pd 
import numpy as np 
import joblib 
import os 
from sklearn.linear_model import LinearRegression 
from sklearn.ensemble import RandomForestRegressor 
from sklearn.ensemble import GradientBoostingRegressor 
from sklearn.metrics import ( mean_absolute_error, mean_squared_error, r2_score )

In [66]:
X_train = pd.read_csv( "data/X_train_scaled.csv" ) 
X_test = pd.read_csv( "data/X_test_scaled.csv" ) 
y_train = pd.read_csv( "data/y_train.csv" ).squeeze() 
y_test = pd.read_csv( "data/y_test.csv" ).squeeze()

In [67]:
models = { 
    "Linear Regression": LinearRegression(), 
    "Random Forest": RandomForestRegressor( 
        n_estimators=200, 
        random_state=42, 
        n_jobs=-1 
    ), 
    "Gradient Boosting": GradientBoostingRegressor( 
        n_estimators=200, 
        learning_rate=0.05, 
        max_depth=3, 
        random_state=42 
    )
}

In [68]:
os.makedirs( "models", exist_ok=True )

In [69]:
results = {} 
trained_models = {} 
for name, model in models.items(): 
    print("\n" + "=" * 50) 
    print(f"Training {name}") 
    print("=" * 50) 
    # Train 
    model.fit( X_train, y_train ) 
    # Predict 
    predictions = model.predict( X_test ) 
    # Metrics 
    mae = mean_absolute_error( y_test, predictions ) 
    mse = mean_squared_error( y_test, predictions ) 
    rmse = np.sqrt(mse) 
    r2 = r2_score( y_test, predictions ) 
    results[name] = { 
        "MAE": mae, 
        "MSE": mse, 
        "RMSE": rmse, 
        "R2": r2 
    } 
    trained_models[name] = model 
    print(f"MAE : {mae:.4f}") 
    print(f"MSE : {mse:.4f}") 
    print(f"RMSE : {rmse:.4f}") 
    print(f"R2 : {r2:.4f}")


Training Linear Regression
MAE : 53.9336
MSE : 4041.5723
RMSE : 63.5734
R2 : -0.0014

Training Random Forest
MAE : 62.8452
MSE : 6020.4250
RMSE : 77.5914
R2 : -0.4918

Training Gradient Boosting
MAE : 53.9859
MSE : 4052.8173
RMSE : 63.6617
R2 : -0.0042


In [70]:
results_df = pd.DataFrame( results ).T 
results_df.index.name = "Model" 
results_df = results_df.reset_index() 
print("\n========== MODEL COMPARISON ==========") 
print( results_df.to_string( index=False ) )


========== MODEL COMPARISON ==========
            Model       MAE         MSE      RMSE        R2
Linear Regression 53.933648 4041.572316 63.573362 -0.001432
    Random Forest 62.845165 6020.425024 77.591398 -0.491758
Gradient Boosting 53.985894 4052.817310 63.661741 -0.004219


In [71]:
best_model_name = results_df.loc[ 
    results_df["RMSE"].idxmin(), 
    "Model" 
] 
best_model = trained_models[ best_model_name ]

In [72]:
best_rmse = results_df.loc[ 
    results_df["Model"] == best_model_name, 
    "RMSE" 
].values[0] 
best_r2 = results_df.loc[ 
    results_df["Model"] == best_model_name, 
    "R2" 
].values[0]

print("\n========================================") 
print("BEST MODEL") 
print("========================================") 
print( f"Model : {best_model_name}" ) 
print( f"RMSE : {best_rmse:.4f}" ) 
print( f"R2 : {best_r2:.4f}" )


BEST MODEL
Model : Linear Regression
RMSE : 63.5734
R2 : -0.0014


In [73]:
joblib.dump( best_model, "models/best_model.pkl" )

['models/best_model.pkl']

In [74]:
with open( 
    "models/best_model_name.txt", 
    "w" 
) as file: 
    file.write( best_model_name )

In [75]:
results_df.to_csv( 
    "models/model_comparison.csv", 
    index=False 
) 
print("\nBest model saved:") 
print("models/best_model.pkl") 
print("\nBest model name saved:") 
print("models/best_model_name.txt") 
print("\nModel comparison saved:") 
print("models/model_comparison.csv")


Best model saved:
models/best_model.pkl

Best model name saved:
models/best_model_name.txt

Model comparison saved:
models/model_comparison.csv
